In [108]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import re

Итак, мы работаем со следующим датасетом:

In [109]:
df = pd.read_csv('all_hati_andrey.csv')
df

,Ссылка,Заголовок,Цена,Оценка циана,ЖК,Агенство,Общая площадь,Жилая площадь,Площадь кухни,Этаж,...,С мебелью,Проверено в Росреестре,Продаётся с мебелью,О подъезде,Строительная серия,Придомовая территория,Аукцион,Год сдачи,Размер доли,Сделка от Циана
0,https://ekb.cian.ru/sale/flat/330868188/?conte...,"Продается 1-комн. квартира, 31,5 м²",4 500 000 ₽,"4,5 млн ₽",NaN,МЕТРЫ,"31,5 м²",17 м²,8 м²,1 из 9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://ekb.cian.ru/sale/flat/330864580/?conte...,"Продается 1-комн. квартира, 30 м²",4 200 000 ₽,"4,3 млн ₽",NaN,DOMOKOM,30 м²,"17,9 м²",6 м²,2 из 5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://ekb.cian.ru/sale/flat/330863793/?conte...,"Продается 1-комн. квартира, 35,2 м²",5 800 000 ₽,"4,3 млн ₽",ЖК «Школьный»,ЕКБ,"35,2 м²","14,6 м²","10,2 м²",10 из 15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://ekb.cian.ru/sale/flat/329175061/?conte...,"Продается 1-комн. квартира, 38,2 м²",9 300 000 ₽,"7,1 млн ₽",NaN,РИМ,"38,2 м²","11,5 м²","14,9 м²",3 из 29,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://ekb.cian.ru/sale/flat/329141191/?conte...,"Продается 1-комн. квартира, 47,1 м²",7 880 000 ₽,NaN,NaN,Актив недвижимость и инвестиции,"47,1 м²","12,8 м²","22,5 м²",14 из 25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
982,https://ekb.cian.ru/sale/flat/330269965/?conte...,"Продается 2-комн. квартира, 47,6 м²",7 500 000 ₽,"6,8 млн ₽",NaN,NaN,"47,6 м²",29 м²,8 м²,6 из 16,...,1.0,NaN,Да,NaN,NaN,NaN,NaN,NaN,NaN,NaN
983,https://ekb.cian.ru/sale/flat/329915308/?conte...,"Продается 2-комн. квартира, 50 м²",4 350 000 ₽,"4,7 млн ₽",NaN,Новосёл,50 м²,29 м²,7 м²,4 из 5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
984,https://ekb.cian.ru/sale/flat/330372745/?conte...,"Продается 2-комн. квартира, 66,8 м²",7 490 000 ₽,"8,4 млн ₽",NaN,Самолет Плюс Екатеринбург,"66,8 м²","33,6 м²","14,1 м²",1 из 13,...,1.0,NaN,Да,NaN,NaN,NaN,NaN,NaN,NaN,NaN
985,https://ekb.cian.ru/sale/flat/329723834/?conte...,"Продается 2-комн. квартира, 48,7 м²",7 550 000 ₽,"7,0 млн ₽",NaN,NaN,"48,7 м²",NaN,8 м²,4 из 12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [110]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 987 entries, 0 to 986
Data columns (total 47 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Ссылка                           987 non-null    object 
 1   Заголовок                        982 non-null    object 
 2   Цена                             979 non-null    object 
 3   Оценка циана                     772 non-null    object 
 4   ЖК                               253 non-null    object 
 5   Агенство                         520 non-null    object 
 6   Общая площадь                    979 non-null    object 
 7   Жилая площадь                    839 non-null    object 
 8   Площадь кухни                    959 non-null    object 
 9   Этаж                             979 non-null    object 
 10  Год постройки                    954 non-null    float64
 11  Метро                            884 non-null    object 
 12  Адрес                 

Перед нами стоит задача каким-то образом обрабатывать описания объявлений, а именно доставать оттуда признаки, которых нет в самом датасете. Вот как выглядит рандомное описание:

In [111]:
df['Описание'].sample().tolist()[0]

'Арт. 133432103 Продается 2-комнатная квартира с функциональной планировкой. \n\nКлючевые особенности:\n Площадь 54,2 кв. м  2 отдельные комнаты, кухня, 2 с/у\n 13/15 этаж  тихо и комфортно  \nРемонт: от застройщика, заезжай и живи.  \nКухня 12 кв. м \n Раздельный санузел  дополнительное удобство  \n Балкон с видом на лес\n\n Расположение и инфраструктура:  \n Школы и детские сады  в шаговой доступности (школы  123,  16,  19, детские сады 45, 119).  \n Транспорт  рядом остановки общественного транспорта (автобусы, маршрутки), удобный выезд на ул. Амундсена и в центр.  \n Магазины и ТЦ  торговые центры "Академический", "МЕГА", "Краснолесья". Супермаркеты "Пятерочка", "Магнит", "Лента".  \n  Поликлиники и аптеки  детская и взрослая поликлиники, аптеки в пешей доступности.\n Парки и спорт  скверы, велодорожки, спортивные площадки, рядом парк "Академический". \n  Внутри двора детская площадка, весь двор и подъезд - подключены к видеонаблюдению, есть возможность просмотра livе-трансляции!\n

Обрабатывать описания мы будем в первую очередь с помощью регулярных выражений. Напишем функцию, которая ищет в тексте ключевые слова и записывает их в отдельные колонки (все признаки будут бинарными, то есть, например, либо консьерж есть, либо его нет - 0 либо 1):

In [112]:
def extract_features(text):
  features = {'has_round_the_clock_security': 0, # безопасность
              'has_terrace': 0, # терраса
              'has_quiet_neighbors': 0, # тихие соседи
              'has_garderobe': 0, # гардеробная
              'has_heated_floor': 0, # теплый пол
              'has_conditioner': 0, # кондиционер
              'has_smart_home': 0, # умный дом
              'has_water_heater': 0, # водонагреватель
              'has_children_playground': 0, # детская площадка
              'has_green_yard': 0, # зеленый двор
              'has_quiet_location': 0, # тихое место
              'has_own_parking_place': 0, # выделенные места для парковки
              'has_underground_parking': 0, # подземная парковка
              'has_school_nearby': 0, # школа/лицей/гимназия рядом
              'has_kindergarten_nearby': 0, # детсад рядом
              'has_shop_nearby': 0, # магазины/ТЦ рядом
              'has_transport_stop': 0, # остановка/метро рядом
              'has_park_nearby': 0, # парк/сквер/лесопарк рядом
              'has_sports_facility': 0, # спорт рядом
              'has_fridge': 0, # холодильник
              'has_washing_machine': 0, # стиралка
              'has_furniture': 0, # мебель
              'has_intercom': 0, # домофон
              'has_clean_entrance': 0, # чистый подъезд
              'has_large_bathroom': 0} # большой туалет


  if not isinstance(text,str) or not text.strip():
    return features

  text = text.lower()

  if re.search(r'круглосуточная охрана|консьерж|видеонаблюдение|охраняемая', text):
    features['has_round_the_clock_security'] = 1

  if re.search(r'террас', text):
    features['has_terrace'] = 1

  if re.search(r'тихие соседи|соседи тихие|адекватные соседи|взрослые люди', text):
    features['has_quiet_neighbors'] = 1

  if re.search(r'гардеробн|шкаф-купе', text):
    features['has_garderobe'] = 1

  if re.search(r'тёплый пол|теплый пол', text):
    features['has_heated_floor'] = 1

  if re.search(r'кондиционер|сплит-система', text):
    features['has_conditioner'] = 1

  if re.search(r'умный дом|smart home', text):
    features['has_smart_home'] = 1

  if re.search(r'водонагреватель|бойлер', text):
    features['has_water_heater'] = 1

  if re.search(r'детская площадка|игровая зона|детский городок', text):
    features['has_children_playground'] = 1

  if re.search(r'зеленый массив|зелёный массив|лесопарк|зеленая зона', text):
    features['has_green_yard'] = 1

  if re.search(r'удален от дороги|меньше шума|тихий район|спальный район', text):
    features['has_quiet_location'] = 1

  if re.search(r'выделенные места для парковки|парковочные места|бесплатная парковка', text):
    features['has_own_parking_place'] = 1

  if re.search(r'подземная парковка|подземный паркинг', text):
    features['has_underground_parking'] = 1

  if re.search(r'школ[аы]|лицей|гимнази', text):
    features['has_school_nearby'] = 1

  if re.search(r'детский сад|детсад|садик', text):
    features['has_kindergarten_nearby'] = 1

  if re.search(r'магазин|торговый центр|тц|гипермаркет|супермаркет|пятерочка|магнит', text):
    features['has_shop_nearby'] = 1

  if re.search(r'метро|остановка|автобус|трамвай|транспортная развязка', text):
    features['has_transport_stop'] = 1

  if re.search(r'парк|сквер|набережная|аллеи|лес\s|лесопарк', text):
    features['has_park_nearby'] = 1

  if re.search(r'фок|спортивный комплекс|горнолыжк|термальный комплекс|бассейн', text):
    features['has_sports_facility'] = 1

  if re.search(r'холодильник', text):
    features['has_fridge'] = 1

  if re.search(r'стиральная машина|стиралка', text):
    features['has_washing_machine'] = 1

  if re.search(r'мебель|диван|кровать|шкаф|кухонный гарнитур|стол', text):
    features['has_furniture'] = 1

  if re.search(r'домофон|видеодомофон', text):
    features['has_intercom'] = 1

  if re.search(r'чистый подъезд|подъезд чистый|ремонт в подъезде|ухоженный подъезд', text):
    features['has_clean_entrance'] = 1

  if re.search(r'просторный санузел|большой санузел|раздельный санузел', text):
    features['has_large_bathroom'] = 1

  return features

In [113]:
bin_features_df = df['Описание'].apply(extract_features)
df_bin = pd.DataFrame(bin_features_df.tolist())
df_bin.describe().loc['mean']

,mean
has_round_the_clock_security,0.159068
has_terrace,0.010132
has_quiet_neighbors,0.018237
has_garderobe,0.132725
has_heated_floor,0.037487
has_conditioner,0.057751
has_smart_home,0.013171
has_water_heater,0.029382
has_children_playground,0.070922
has_green_yard,0.044580


Однако видим, что многие признаки встречаются в датасете слишком редко (например, has_quiet_location, has_smart_home, has_heated_floor) - это пллохо, потому что будущая модель будет воспринимать это как шум, и это ухудшит ее эффективность. Удалим признаки, которые слишком редко бывают =1 (встречаются < 5% случаев):

In [114]:
df_bin = df_bin.drop(['has_quiet_location', 'has_smart_home', 'has_terrace', 'has_water_heater'], axis=1)

In [115]:
df = pd.concat([df,df_bin], axis=1)
df

,Ссылка,Заголовок,Цена,Оценка циана,ЖК,Агенство,Общая площадь,Жилая площадь,Площадь кухни,Этаж,...,has_shop_nearby,has_transport_stop,has_park_nearby,has_sports_facility,has_fridge,has_washing_machine,has_furniture,has_intercom,has_clean_entrance,has_large_bathroom
0,https://ekb.cian.ru/sale/flat/330868188/?conte...,"Продается 1-комн. квартира, 31,5 м²",4 500 000 ₽,"4,5 млн ₽",NaN,МЕТРЫ,"31,5 м²",17 м²,8 м²,1 из 9,...,0,0,0,0,0,0,0,0,0,0
1,https://ekb.cian.ru/sale/flat/330864580/?conte...,"Продается 1-комн. квартира, 30 м²",4 200 000 ₽,"4,3 млн ₽",NaN,DOMOKOM,30 м²,"17,9 м²",6 м²,2 из 5,...,0,1,0,0,0,0,0,0,0,0
2,https://ekb.cian.ru/sale/flat/330863793/?conte...,"Продается 1-комн. квартира, 35,2 м²",5 800 000 ₽,"4,3 млн ₽",ЖК «Школьный»,ЕКБ,"35,2 м²","14,6 м²","10,2 м²",10 из 15,...,1,1,1,1,1,1,1,0,1,0
3,https://ekb.cian.ru/sale/flat/329175061/?conte...,"Продается 1-комн. квартира, 38,2 м²",9 300 000 ₽,"7,1 млн ₽",NaN,РИМ,"38,2 м²","11,5 м²","14,9 м²",3 из 29,...,1,1,1,0,0,0,0,0,0,0
4,https://ekb.cian.ru/sale/flat/329141191/?conte...,"Продается 1-комн. квартира, 47,1 м²",7 880 000 ₽,NaN,NaN,Актив недвижимость и инвестиции,"47,1 м²","12,8 м²","22,5 м²",14 из 25,...,1,0,1,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
982,https://ekb.cian.ru/sale/flat/330269965/?conte...,"Продается 2-комн. квартира, 47,6 м²",7 500 000 ₽,"6,8 млн ₽",NaN,NaN,"47,6 м²",29 м²,8 м²,6 из 16,...,1,1,1,0,0,0,1,0,0,0
983,https://ekb.cian.ru/sale/flat/329915308/?conte...,"Продается 2-комн. квартира, 50 м²",4 350 000 ₽,"4,7 млн ₽",NaN,Новосёл,50 м²,29 м²,7 м²,4 из 5,...,1,1,0,0,0,0,0,0,0,1
984,https://ekb.cian.ru/sale/flat/330372745/?conte...,"Продается 2-комн. квартира, 66,8 м²",7 490 000 ₽,"8,4 млн ₽",NaN,Самолет Плюс Екатеринбург,"66,8 м²","33,6 м²","14,1 м²",1 из 13,...,1,1,0,0,0,0,0,0,0,0
985,https://ekb.cian.ru/sale/flat/329723834/?conte...,"Продается 2-комн. квартира, 48,7 м²",7 550 000 ₽,"7,0 млн ₽",NaN,NaN,"48,7 м²",NaN,8 м²,4 из 12,...,1,1,1,0,0,0,1,0,0,0


Неплохо, у нас добавился 21 бинарный признак

Однако регулярные выражения не в состоянии охватить весь смысл текста, поэтому для этого мы также прогоним текст через локальную модель ruBERT-tiny2. Это легкая и быстрая (содержит достаточно мало параметров относительно других моделей и занимает мало оперативной памяти, что важно в рамках нашего проекта) нейросетевая модель для обработки текстов в т.ч. на русском языке. При всем при этом она имеет высокую точность, которой с головой хватит для наших 6000 описаний

Приступим к прогонке. Загружаем модель:

In [116]:
model = SentenceTransformer('cointegrated/rubert-tiny2')

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Подготавливаем наши тексты для дальнейшей передачи модели:

In [117]:
descriptions = df['Описание'].fillna('').astype(str).tolist()

Получаем эмбеддинги, каждый из которых состоит из 312 чисел:

In [118]:
embeddings = model.encode(descriptions,
                          show_progress_bar=True,
                          batch_size=64,
                          normalize_embeddings=True)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Готово, добавляем в датафрейм:

In [119]:
embed_cols = [f'desc_emb_{i}' for i in range(embeddings.shape[1])]
df_embed = pd.DataFrame(embeddings, columns=embed_cols)
df = pd.concat([df, df_embed], axis=1)

In [120]:
df.shape

(987, 380)

Целых 380 признаков получилось, включая эмбеддинги. Может показаться, что этого слишком много для нашей будущей модели, и возможно она переобучится. Однако у нас будет полносвязная нейронная сеть, и такие модели не боятся мультиколлинеарности, а также отношение числа наблюдений (6000) к числу признаков (380) примерно равно 16, что практически идеально для полносвязной сети (доллжно быть 10-20 наблюдений на один признак)